In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [2]:
model_path = "./models/gemma-4-E4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [3]:
adapter_path = "./adapters/tool"
model_with_adapter = PeftModel.from_pretrained(model, adapter_path)

In [4]:
def generate_lyrics(model, tokenizer, prompt="Write song lyrics.\n\n", max_new_tokens=512, min_new_tokens=200, temperature=0.9, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        repetition_penalty=1.1,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
print(generate_lyrics(model_with_adapter, tokenizer))

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Write song lyrics.

I'm not the one to judge
But I have a suggestion for you
You should go home and see your son
Tell him, "Daddy misses you"


And if that means something to you
We can start by going fishing soon
Maybe fly fish in Montana
Let you feel alive again soon


Take my advice like an order
Go home and see your son
If he doesn't wanna play then
Watch some TV or a movie on
Then we'll find a lake
Get your license and take me down there


If it don't work then try this:
Just do what you did before
That was right here and now when
He had no reason to die at all

We need to change things around again
Bring people together so they can
Celebrate life over a river and water
A bottle of wine or two maybe

If we want to move past our fear of each other
This is a good place to start


Buy a ticket, get out there, listen
Trust your instincts, remember your faith in


Change things around again
It is time to celebrate being alive
Learn to speak from open hearts and minds
Give up judging ot

In [6]:
print(model_with_adapter.active_adapters)

['default']


In [7]:
import os
os.listdir("./adapters/gojira")

['checkpoint-100',
 'checkpoint-5',
 'README.md',
 'checkpoint-40',
 'adapter_config.json',
 'checkpoint-25',
 'checkpoint-60',
 'checkpoint-10',
 'checkpoint-15',
 'adapter_model.safetensors',
 'checkpoint-20',
 'checkpoint-80']

In [8]:
import json
with open("./adapters/gojira/adapter_config.json") as f:
      print(json.dumps(json.load(f), indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "./models/gemma-4-E4B",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.1,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "peft_version": "0.19.1",
  "qalora_group_size": 16,
  "r": 8,
  "rank_pattern": {},
  "revision": null,
  "target_modules": "model\\.language_model\\.layers\\.\\d+\\.(self_attn\\.(q|k|v|o)_proj|mlp\\.(gate|up|down)_proj)",
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trainable_token_indices": null,
  "u

In [9]:
import trl
print(trl.__version__)

1.4.0
